# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [30]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Perform Train Test Split**

In [3]:
# Tran Test Split

features = df.drop(columns=["Transported"])
target = df["Transported"]

x_train, x_test, y_train, y_test = train_test_split(features, target, test_size = 0.2, random_state = 42)

In [4]:
# Cabin simplification

x_train["Cabin"] = x_train["Cabin"].str[0]
x_test["Cabin"] = x_test["Cabin"].str[0]

In [5]:
# Drop PassengerId and Name columns

x_train = x_train.drop(columns=["PassengerId", "Name"])
x_test = x_test.drop(columns=["PassengerId", "Name"])

Now perform the same as before:

- Null values filling
- Feature Scaling
- Feature Selection


In [6]:
# Encoding and Null values info

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.2+ MB


In [7]:
# Encoding

cat_cols = ["HomePlanet", "Cabin", "Destination"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

train_encoded = encoder.fit_transform(x_train[cat_cols])

test_encoded = encoder.transform(x_test[cat_cols])

train_encoded_df = pd.DataFrame(train_encoded, columns=encoder.get_feature_names_out(cat_cols))
test_encoded_df = pd.DataFrame(test_encoded, columns=encoder.get_feature_names_out(cat_cols))

In [8]:
# Drop categorical features after encoding

x_train = x_train.drop(columns=cat_cols)
x_test = x_test.drop(columns=cat_cols)

In [9]:
# Concadenating Encoded data

x_train = pd.concat([x_train.reset_index(drop=True), train_encoded_df.reset_index(drop=True)], axis=1)
x_test = pd.concat([x_test.reset_index(drop=True), test_encoded_df.reset_index(drop=True)], axis=1)

In [10]:
# Fill NaN values fro Boolean features

x_train["CryoSleep"] = x_train["CryoSleep"].fillna(False).astype(int)
x_test["CryoSleep"] = x_test["CryoSleep"].fillna(False).astype(int)

x_train["VIP"] = x_train["VIP"].fillna(False).astype(int)
x_test["VIP"] = x_test["VIP"].fillna(False).astype(int)

In [11]:
# Changing type for boolean features

x_train["CryoSleep"] = x_train["CryoSleep"].astype(int)
x_test["CryoSleep"] = x_test["CryoSleep"].astype(int)

x_train["VIP"] = x_train["VIP"].astype(int)
x_test["VIP"] = x_test["VIP"].astype(int)

y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [12]:
x_train

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Cabin_nan,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_nan
0,0,28.0,0,0.0,55.0,0.0,656.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,0,17.0,0,0.0,1195.0,31.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1,28.0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0,20.0,0,NaN,2.0,289.0,976.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1,36.0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6949,0,18.0,0,14.0,2.0,144.0,610.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
6950,0,50.0,0,690.0,0.0,30.0,762.0,428.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6951,0,22.0,0,158.0,0.0,476.0,0.0,26.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6952,0,34.0,0,379.0,0.0,1626.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [13]:
# Filling NaN values with the mean

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["Age"]])
x_train["Age"] =imputer.transform(x_train[["Age"]])
x_test["Age"] =imputer.transform(x_test[["Age"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["RoomService"]])
x_train["RoomService"] =imputer.transform(x_train[["RoomService"]])
x_test["RoomService"] =imputer.transform(x_test[["RoomService"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["FoodCourt"]])
x_train["FoodCourt"] =imputer.transform(x_train[["FoodCourt"]])
x_test["FoodCourt"] =imputer.transform(x_test[["FoodCourt"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["ShoppingMall"]])
x_train["ShoppingMall"] =imputer.transform(x_train[["ShoppingMall"]])
x_test["ShoppingMall"] =imputer.transform(x_test[["ShoppingMall"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["Spa"]])
x_train["Spa"] =imputer.transform(x_train[["Spa"]])
x_test["Spa"] =imputer.transform(x_test[["Spa"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["VRDeck"]])
x_train["VRDeck"] =imputer.transform(x_train[["VRDeck"]])
x_test["VRDeck"] =imputer.transform(x_test[["VRDeck"]])

In [14]:
x_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CryoSleep                  6954 non-null   int64  
 1   Age                        6954 non-null   float64
 2   VIP                        6954 non-null   int64  
 3   RoomService                6954 non-null   float64
 4   FoodCourt                  6954 non-null   float64
 5   ShoppingMall               6954 non-null   float64
 6   Spa                        6954 non-null   float64
 7   VRDeck                     6954 non-null   float64
 8   HomePlanet_Earth           6954 non-null   float64
 9   HomePlanet_Europa          6954 non-null   float64
 10  HomePlanet_Mars            6954 non-null   float64
 11  HomePlanet_nan             6954 non-null   float64
 12  Cabin_A                    6954 non-null   float64
 13  Cabin_B                    6954 non-null   float64
 14  Cab

In [15]:
# Feature selection

columns_to_drop = ["Destination_TRAPPIST-1e", "HomePlanet_Earth", "HomePlanet_Europa", "Cabin_T", "Destination_PSO J318.5-22", "Cabin_A"]

x_train = x_train.drop(columns=columns_to_drop, errors="ignore")
x_test = x_test.drop(columns=columns_to_drop, errors="ignore")


In [16]:
# Feature scaling: Standarization

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_test_scaled = scaler.transform(x_test)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [26]:
bag_clf = BaggingClassifier(random_state=42)
bag_clf.fit(x_train_scaled, y_train)
print("Bagging Accuracy:", accuracy_score(y_test, bag_clf.predict(x_test_scaled)))

Bagging Accuracy: 0.78205865439908


- Random Forests

In [27]:
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(x_train_scaled, y_train)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_clf.predict(x_test_scaled)))

Random Forest Accuracy: 0.7878090856814262


- Gradient Boosting

In [28]:
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(x_train_scaled, y_train)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, gb_clf.predict(x_test_scaled)))

Gradient Boosting Accuracy: 0.7901092581943646


- Adaptive Boosting

In [29]:
ada_clf = AdaBoostClassifier(random_state=42)
ada_clf.fit(x_train_scaled, y_train)
print("AdaBoost Accuracy:", accuracy_score(y_test, ada_clf.predict(x_test_scaled)))

AdaBoost Accuracy: 0.7699827487061529


Which model is the best and why?

In [33]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train_scaled, y_train)
y_pred_knn = knn_model.predict(x_test_scaled)
knn.score(x_test_scaled, y_test) 

0.7699827487061529

In [ ]:
The best model is the Gradient Boosting because it has the highest score.